In [7]:
%pip install rank_bm25
%pip install -U langchain langchain-community langchain-huggingface langchain-chroma rank_bm25 python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

from dotenv import load_dotenv

load_dotenv()

C:\Users\senth\AppData\Local\Temp\ipykernel_5864\2430581772.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


True

In [11]:
# ──────────────────────────────────────────────────────────────────
# SETUP: Create our sample company data
# ──────────────────────────────────────────────────────────────────

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [12]:
# Convert to Document objects for LangChain
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

print("Sample Data:")
for i, chunk in enumerate(chunks, 1):
    print(f"{i}. {chunk}")

print("\n" + "="*80)

Sample Data:
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java pr

 
SETUP: Create the three types of retrievers

# 1. Vector Retriever (Semantic Search/Dense Retrieval)


In [13]:
print("Setting up Vector Retriever...")
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

Setting up Vector Retriever...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Test semantic search
test_query = "space exploration company" #works in vector search but wouldn't work with keyword search

print(f"Testing: '{test_query}'")
test_docs = vector_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing: 'space exploration company'
Found: SpaceX develops Starship rockets for Mars missions.
Found: Google is a large technology company with global operations.


# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)


In [15]:
# 2. BM25 Retriever (Keyword Search)
print("Setting up BM25 Retriever...")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3

Setting up BM25 Retriever...


In [16]:
# Test exact keyword matching
# test_query = "space exploration company"
test_query = "Cybertruck"
# test_query = "Tesla"

print(f"Testing: '{test_query}'")
test_docs = bm25_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing: 'Cybertruck'
Found: Cybertruck reservations exceeded company expectations.
Found: Tesla Cybertruck production ramp begins in 2024.
Found: Orange juice sales increased during winter months.


# 3. Hybrid Retriever (Combination)

In [17]:
#  3. Hybrid Retriever (Combination)
print("Setting up Hybrid Retriever...")
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]  # Equal weight to vector and keyword search
)

print("Setup complete!\n")

Setting up Hybrid Retriever...
Setup complete!



In [18]:
# Query 1: Mixed semantic and exact terms

# Vector search understands "purchase cost" semantically
# BM25 search finds exact "7.5 billion" 
# Hybrid should combine both strengths for best result
test_query = "purchase cost 7.5 billion"

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print()

print("Query 1 shows how hybrid finds exact financial info using both semantic understanding and keyword matching")


1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Microsoft is a large technology company with global operations.
3. Orange County reported new housing developments.
4. Orange juice sales increased during winter months.

Query 1 shows how hybrid finds exact financial info using both semantic understanding and keyword matching


In [19]:
# Query 2: Semantic concept + specific product name  

# Vector search understands "electric vehicle manufacturing"
# BM25 search finds exact "Cybertruck"
# Hybrid gets the best of both worlds

test_query = "electric vehicle manufacturing Cybertruck"

retrieved_chunks = hybrid_retriever.invoke(test_query)

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print()

print("Query 2 demonstrates combining product-specific terms with broader concepts")



1. Tesla Cybertruck production ramp begins in 2024.
2. Cybertruck reservations exceeded company expectations.
3. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.

Query 2 demonstrates combining product-specific terms with broader concepts


In [20]:
# Query 3: Where neither alone would be perfect

# "Company performance" is semantic, "Tesla" is exact keyword
# Hybrid should find the most relevant Tesla performance info

test_query = "company performance Tesla"

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print()

print("Query 3 shows how hybrid handles mixed semantic/keyword queries better than either approach alone")

1. Tesla Tesla Tesla financial quarterly results improved significantly.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Cybertruck reservations exceeded company expectations.

Query 3 shows how hybrid handles mixed semantic/keyword queries better than either approach alone


In [21]:
# Combine the query and the relevant document contents
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI


combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

# Create a ChatOpenAI model
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# Define the messages for the model
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = model.invoke(messages)

# Display the full result and content only
print("\n--- Generated Response ---")
# print("Full result:")
# print(result)
print("Content only:")
print(result.content)


C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



--- Generated Response ---
Content only:
[{'type': 'text', 'text': "Based on the provided documents, Tesla's company performance highlights include:\n\n* **Financial Performance:** Tesla's financial quarterly results improved significantly and were reported as strong.\n* **Market Leadership:** Tesla continues to lead in electric vehicles.\n* **Product Demand:** Cybertruck reservations exceeded the company's expectations.\n* **Growth/Expansion:** Tesla announced new manufacturing facilities.", 'extras': {'signature': 'Et4MCtsMARFNMg/BxMFz35ZpcJDoYlNriC7kxUG0IwwETrvtYsOo8XsPWsBOttSXMTlHlWh7hPVD3VtQyDMtGLm9ECBTxfj7ABiQabrnEwyM2TLhCWgLIAmInc2g6ZN70iQbeQZZs81FGtC0SWnfKnL3eDAb/x1Jmi8ERXqYqS5tJkN6c6M7aomjw4kE+L8oK9LLOTo4nqXm5Yz4sWo7DmPVxbHTk/CBcT61JFXbx7iC5gbewyuEs6cO+9JLY5/FCzD8L6vJc/MYDYIJmVX2hEq0G3BIyJgKBDEC2QJUa7fJH5A0V9F6eWnX9I3RHXvJLkubZhsPb0j1i7x3g7QVVDYWKulEF8rmMiQbPm/WbkhaWlGW7NPRrJfr97sXSN4TEukAkJFdugRZ7ml1OxSIoDYlC0wo/5IN8rjXh+sO97RTB9/0SJtF1WSrPRguuS60D1IDcw5ERRx4KARoArhlmk0FtTEg